### 앙상블 리트리버 ( Ensemble Retriever )

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [4]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook"
]

bm25_retriever = BM25Retriever.from_texts(doc_list)
bm25_retriever.k = 1

embedding = OpenAIEmbeddings()

faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding
)

faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k":1})

ensemble_retriever = EnsembleRetriever(
    retrievers= [bm25_retriever, faiss_retriever],
    weights= [0.7, 0.3]
)

In [5]:
# ensemble_retriever 객체릐 get_relevant_documents() 메서드를 호출하여 관련성 높은 문서 검색

# 검색 결과 문서를 가져옴
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content : {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content : {doc.page_content}")
    print()


print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content : {doc.page_content}")
    print()

[Ensemble Retriever]
Content : Apple is my favorite company

Content : I like apples

[BM25 Retriever]
Content : Apple is my favorite company

[FAISS Retriever]
Content : I like apples



In [6]:
# 검색 결과 문서를 가져옵니다.
query = "Apple company makes my favorite iphone"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력합니다.
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apple's iphone

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apple's iphone



- 런타임 Config 변경

In [9]:
from langchain_core.runnables import ConfigurableField

ensemble_retriever = EnsembleRetriever(
    # 리트리버 목록을 설정
    retrievers= [bm25_retriever, faiss_retriever]
).configurable_fields(
    weights= ConfigurableField(
    # 검색 매개변수의 고유 식별자를 설정
    id= "ensemble_weights",
    # 검색 매개변수의 이름을 설정
    name= "Ensemble Weights",
    # 검색 매개변수에 대한 설명을 작성
    description= "Ensemble Weights"
    )
)

In [11]:
# 검색시 config 매개변수를 통해 검색 설정 지정

config = {"configurable": {"ensemble_weights": [1, 0]}}

docs = ensemble_retriever.invoke("my favorite fruit is apple", config= config)
docs

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='4dc3e13f-e0a1-4ef5-b0d9-35650129e1a1', metadata={}, page_content='I like apples')]

In [12]:
config = {"configurable": {"ensemble_weights": [0, 1]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # 검색 결과인 docs를 출력합니다.

[Document(id='4dc3e13f-e0a1-4ef5-b0d9-35650129e1a1', metadata={}, page_content='I like apples'),
 Document(metadata={}, page_content='Apple is my favorite company')]